In [1]:
import torch

print("torch version:", torch.__version__)
print("CUDA 可用嗎：", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 名稱：", torch.cuda.get_device_name(0))
    print("CUDA version in torch:", torch.version.cuda)
    device = torch.device("cuda")
else:
    print("目前還是沒有用到 CUDA GPU")
    device = torch.device("cpu")

print("使用裝置：", device)

torch version: 2.11.0+cu128
CUDA 可用嗎： True
GPU 名稱： NVIDIA GeForce RTX 5060 Ti
CUDA version in torch: 12.8
使用裝置： cuda


In [2]:
import pandas as pd

TEXT_COL = "Text"
LABEL_COL = "Score"

TRAIN_FILES = [
    "Dataset.xlsx",
    "Reviews.xlsx"
]

train_dfs = []

for file in TRAIN_FILES:
    df = pd.read_excel(file)

    # 去掉欄位名稱前後空白，避免 "Text " 這種問題
    df.columns = df.columns.astype(str).str.strip()

    print(f"\n檔案：{file}")
    print("欄位：", list(df.columns))

    # 只保留 Text 和 Score
    df = df[[TEXT_COL, LABEL_COL]].dropna()

    print("資料大小：", df.shape)
    train_dfs.append(df)

train_df = pd.concat(train_dfs, ignore_index=True)

print("\n合併後訓練資料：", train_df.shape)
display(train_df.head())


檔案：Dataset.xlsx
欄位： ['Product_name', 'Price', 'Score', 'Text', 'Summary']
資料大小： (363239, 2)

檔案：Reviews.xlsx
欄位： ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']
資料大小： (568454, 2)

合併後訓練資料： (931693, 2)


,Text,Score
0,Simply awesome,5
1,Worth the money . Desert Cooler live up to the...,4
2,Worth every penny,5
3,Fabulous!,5
4,Nice product,4


In [3]:
TEST_FILES = {
    "women": "women reviews on clothes on Amazon.xlsx",
    "walmart": "marketing_sample_for_walmart_com-walmart_product_reviews__20200401_20200630__30k_data.xlsx"
}

test_dfs = {}

for name, file in TEST_FILES.items():
    df = pd.read_excel(file)

    df.columns = df.columns.astype(str).str.strip()

    print(f"\n測試資料：{name}")
    print("檔案：", file)
    print("欄位：", list(df.columns))

    df = df[[TEXT_COL, LABEL_COL]].dropna()

    test_dfs[name] = df

    print("資料大小：", df.shape)
    display(df.head())


測試資料：women
檔案： women reviews on clothes on Amazon.xlsx
欄位： ['Unnamed: 0', 'Clothing ID', 'Age', 'Title', 'Text', 'Score', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name']
資料大小： (6296, 2)


,Text,Score
0,Absolutely wonderful - silky and sexy and comf...,4
1,Love this dress! it's sooo pretty. i happene...,5
2,I had such high hopes for this dress and reall...,3
3,"I love, love, love this jumpsuit. it's fun, fl...",5
4,This shirt is very flattering to all due to th...,5



測試資料：walmart
檔案： marketing_sample_for_walmart_com-walmart_product_reviews__20200401_20200630__30k_data.xlsx
欄位： ['Uniq Id', 'Crawl Timestamp', 'Pageurl', 'Website', 'Title', 'Score', 'Text', 'Reviewer Name', 'Review Upvotes', 'Review Downvotes', 'Verified Purchaser', 'Recommended Purchase', 'Review Date', 'Five Star', 'Four Star', 'Three Star', 'Two Star', 'One Star']
資料大小： (24617, 2)


,Text,Score
0,One star for looking nice. Thats it. After try...,1
1,Love this phone so far have had it almost a mo...,4
2,This TV is absolutely fantastic. This is my th...,4
3,"Refurb, good shape, good price, does what I wa...",5
4,Very nice tablet! Looks brand new. Fired right...,5


In [4]:
import pandas as pd
import re

TEXT_COL = "Text"
LABEL_COL = "Score"

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_review_df(df, name="dataset"):
    df = df[[TEXT_COL, LABEL_COL]].dropna().copy()

    print(f"\n===== 清理前：{name} =====")
    print("資料大小：", df.shape)
    print("欄位：", list(df.columns))

    df[TEXT_COL] = df[TEXT_COL].apply(clean_text)

    # Score 強制轉成數字，不能轉的變 NaN
    df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")

    before = len(df)
    df = df.dropna(subset=[LABEL_COL]).copy()
    print("刪除非數字 Score 筆數：", before - len(df))

    # 只保留 1~5 分
    before = len(df)
    df = df[df[LABEL_COL].between(1, 5)].copy()
    print("刪除非 1~5 分筆數：", before - len(df))

    # 轉成整數
    df[LABEL_COL] = df[LABEL_COL].astype(int)

    # 建立 label：Score 1~5 → label 0~4
    df["label"] = df[LABEL_COL] - 1

    print(f"===== 清理後：{name} =====")
    print("資料大小：", df.shape)
    print("欄位：", list(df.columns))
    print("Score 分布：")
    print(df[LABEL_COL].value_counts().sort_index())

    return df

In [5]:
train_df = clean_review_df(train_df, name="train")

for name in test_dfs:
    test_dfs[name] = clean_review_df(test_dfs[name], name=name)

print("train_df 欄位：", train_df.columns)

for name in test_dfs:
    print(name, test_dfs[name].columns)


===== 清理前：train =====
資料大小： (931693, 2)
欄位： ['Text', 'Score']
刪除非數字 Score 筆數： 6
刪除非 1~5 分筆數： 0
===== 清理後：train =====
資料大小： (931687, 3)
欄位： ['Text', 'Score', 'label']
Score 分布：
Score
1     92345
2     42748
3     74738
4    154906
5    566950
Name: count, dtype: int64

===== 清理前：women =====
資料大小： (6296, 2)
欄位： ['Text', 'Score']
刪除非數字 Score 筆數： 0
刪除非 1~5 分筆數： 0
===== 清理後：women =====
資料大小： (6296, 3)
欄位： ['Text', 'Score', 'label']
Score 分布：
Score
1     211
2     414
3     799
4    1422
5    3450
Name: count, dtype: int64

===== 清理前：walmart =====
資料大小： (24617, 2)
欄位： ['Text', 'Score']
刪除非數字 Score 筆數： 0
刪除非 1~5 分筆數： 0
===== 清理後：walmart =====
資料大小： (24617, 3)
欄位： ['Text', 'Score', 'label']
Score 分布：
Score
1     2971
2     1133
3     1501
4     3893
5    15119
Name: count, dtype: int64
train_df 欄位： Index(['Text', 'Score', 'label'], dtype='object')
women Index(['Text', 'Score', 'label'], dtype='object')
walmart Index(['Text', 'Score', 'label'], dtype='object')


In [6]:
from sklearn.model_selection import train_test_split

train_part, valid_part = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df["label"]
)

print("真正訓練資料：", train_part.shape)
print("驗證資料：", valid_part.shape)

print("\ntrain_part Score 分布：")
print(train_part[LABEL_COL].value_counts().sort_index())

print("\nvalid_part Score 分布：")
print(valid_part[LABEL_COL].value_counts().sort_index())

真正訓練資料： (838518, 3)
驗證資料： (93169, 3)

train_part Score 分布：
Score
1     83111
2     38473
3     67264
4    139415
5    510255
Name: count, dtype: int64

valid_part Score 分布：
Score
1     9234
2     4275
3     7474
4    15491
5    56695
Name: count, dtype: int64


In [7]:
from collections import Counter

MAX_VOCAB_SIZE = 80000
MIN_FREQ = 2

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

def tokenize(text):
    return str(text).split()

counter = Counter()

for text in train_part[TEXT_COL]:
    counter.update(tokenize(text))

most_common = [
    word for word, freq in counter.most_common(MAX_VOCAB_SIZE)
    if freq >= MIN_FREQ
]

word2idx = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}

for word in most_common:
    if word not in word2idx:
        word2idx[word] = len(word2idx)

idx2word = {idx: word for word, idx in word2idx.items()}

VOCAB_SIZE = len(word2idx)

print("詞彙表大小：", VOCAB_SIZE)
print("前 20 個詞：")
print(list(word2idx.items())[:20])

詞彙表大小： 64933
前 20 個詞：
[('<PAD>', 0), ('<UNK>', 1), ('the', 2), ('i', 3), ('and', 4), ('a', 5), ('it', 6), ('to', 7), ('of', 8), ('is', 9), ('this', 10), ('in', 11), ('for', 12), ('my', 13), ('that', 14), ('but', 15), ('you', 16), ('with', 17), ('have', 18), ('not', 19)]


In [8]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

MAX_LEN = 300
BATCH_SIZE = 64

def encode_text(text, word2idx, max_len=MAX_LEN):
    tokens = tokenize(text)

    ids = [word2idx.get(token, word2idx[UNK_TOKEN]) for token in tokens]
    ids = ids[:max_len]

    if len(ids) < max_len:
        ids += [word2idx[PAD_TOKEN]] * (max_len - len(ids))

    return ids


class ReviewDataset(Dataset):
    def __init__(self, df, text_col, label_col, word2idx, max_len):
        self.texts = df[text_col].tolist()
        self.labels = df[label_col].tolist()
        self.word2idx = word2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text_ids = encode_text(
            self.texts[idx],
            self.word2idx,
            self.max_len
        )

        label = self.labels[idx]

        return {
            "input_ids": torch.tensor(text_ids, dtype=torch.long),
            "label": torch.tensor(label, dtype=torch.long)
        }


train_dataset = ReviewDataset(train_part, TEXT_COL, "label", word2idx, MAX_LEN)
valid_dataset = ReviewDataset(valid_part, TEXT_COL, "label", word2idx, MAX_LEN)

# 讓少數分數類別被抽到更多次
label_counts = train_part["label"].value_counts().sort_index()

print("訓練 label 分布：")
print(label_counts)

class_sample_weights = 1.0 / label_counts
sample_weights = train_part["label"].map(class_sample_weights).values
sample_weights = torch.DoubleTensor(sample_weights)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    shuffle=False,
    num_workers=0
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loaders = {}

for name, df in test_dfs.items():
    test_dataset = ReviewDataset(df, TEXT_COL, "label", word2idx, MAX_LEN)

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )

    test_loaders[name] = test_loader

print("train batches:", len(train_loader))
print("valid batches:", len(valid_loader))

for name, loader in test_loaders.items():
    print(f"{name} test batches:", len(loader))

訓練 label 分布：
label
0     83111
1     38473
2     67264
3    139415
4    510255
Name: count, dtype: int64
train batches: 13102
valid batches: 1456
women test batches: 99
walmart test batches: 385


In [9]:
import torch.nn as nn
import torch.nn.functional as F

class BiLSTMAttentionClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim,
        hidden_dim,
        num_classes,
        pad_idx,
        num_layers=2,
        dropout=0.5
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx
        )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.attention = nn.Linear(hidden_dim * 2, 1)

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, input_ids):
        mask = input_ids != 0

        embedded = self.embedding(input_ids)
        lstm_out, _ = self.lstm(embedded)

        attention_scores = self.attention(lstm_out).squeeze(-1)
        attention_scores = attention_scores.masked_fill(~mask, -1e9)

        attention_weights = torch.softmax(attention_scores, dim=1)

        context = torch.sum(lstm_out * attention_weights.unsqueeze(-1), dim=1)
        context = self.dropout(context)

        logits = self.classifier(context)

        return logits

In [10]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("使用裝置：", device)

if torch.cuda.is_available():
    print("GPU：", torch.cuda.get_device_name(0))

EMBED_DIM = 256
HIDDEN_DIM = 256
NUM_CLASSES = 5
NUM_LAYERS = 2
DROPOUT = 0.5

model = BiLSTMAttentionClassifier(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
    pad_idx=word2idx[PAD_TOKEN],
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(device)

print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("總參數量：", total_params)
print("可訓練參數量：", trainable_params)
print("模型裝置：", next(model.parameters()).device)

使用裝置： cuda
GPU： NVIDIA GeForce RTX 5060 Ti
BiLSTMAttentionClassifier(
  (embedding): Embedding(64933, 256, padding_idx=0)
  (lstm): LSTM(256, 256, num_layers=2, batch_first=True, dropout=0.5, bidirectional=True)
  (attention): Linear(in_features=512, out_features=1, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (classifier): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=256, out_features=5, bias=True)
  )
)
總參數量： 19385606
可訓練參數量： 19385606
模型裝置： cuda:0


In [11]:
import torch.nn as nn

criterion = nn.CrossEntropyLoss(
    label_smoothing=0.05
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

print("Loss: CrossEntropyLoss + label_smoothing=0.05")
print("Optimizer: AdamW, lr=5e-4")

Loss: CrossEntropyLoss + label_smoothing=0.05
Optimizer: AdamW, lr=5e-4


In [12]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in tqdm(loader, desc="Training"):
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids)
        loss = criterion(logits, labels)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, macro_f1


def evaluate(model, loader, criterion, device, desc="Evaluating"):
    model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            input_ids = batch["input_ids"].to(device)
            labels = batch["label"].to(device)

            logits = model(input_ids)
            loss = criterion(logits, labels)

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, macro_f1, all_labels, all_preds

In [13]:
EPOCHS = 1
BEST_VALID_F1 = 0
PATIENCE = 5
patience_count = 0

best_model_path = "best_macro_f1_bilstm_attention_model.pt"

history = []

for epoch in range(1, EPOCHS + 1):
    print(f"\n===== Epoch {epoch}/{EPOCHS} =====")

    train_loss, train_acc, train_f1 = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    valid_loss, valid_acc, valid_f1, _, _ = evaluate(
        model,
        valid_loader,
        criterion,
        device,
        desc="Validation"
    )

    scheduler.step(valid_f1)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train Macro-F1: {train_f1:.4f}")
    print(f"Valid Loss: {valid_loss:.4f} | Valid Acc: {valid_acc:.4f} | Valid Macro-F1: {valid_f1:.4f}")

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_f1": train_f1,
        "valid_loss": valid_loss,
        "valid_acc": valid_acc,
        "valid_f1": valid_f1
    })

    if valid_f1 > BEST_VALID_F1:
        BEST_VALID_F1 = valid_f1
        patience_count = 0

        torch.save({
            "model_state_dict": model.state_dict(),
            "word2idx": word2idx,
            "idx2word": idx2word,
            "config": {
                "vocab_size": VOCAB_SIZE,
                "embed_dim": EMBED_DIM,
                "hidden_dim": HIDDEN_DIM,
                "num_classes": NUM_CLASSES,
                "pad_idx": word2idx[PAD_TOKEN],
                "num_layers": NUM_LAYERS,
                "dropout": DROPOUT,
                "max_len": MAX_LEN
            }
        }, best_model_path)

        print("已儲存目前最佳 Macro-F1 模型")
    else:
        patience_count += 1
        print(f"沒有進步，patience = {patience_count}/{PATIENCE}")

        if patience_count >= PATIENCE:
            print("Early stopping")
            break

print("最佳 Valid Macro-F1:", BEST_VALID_F1)


===== Epoch 1/1 =====


Validation: 100%|██████████| 1456/1456 [00:24<00:00, 58.46it/s]


Train Loss: 0.6994 | Train Acc: 0.7820 | Train Macro-F1: 0.7817
Valid Loss: 0.6348 | Valid Acc: 0.8079 | Valid Macro-F1: 0.7561
已儲存目前最佳 Macro-F1 模型
最佳 Valid Macro-F1: 0.7561332521045749


In [14]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import torch

checkpoint = torch.load(best_model_path, map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)

all_test_results = {}

for name, loader in test_loaders.items():
    print(f"\n\n========== 測試資料：{name} ==========")

    test_loss, test_acc, test_f1, true_labels, pred_labels = evaluate(
        model,
        loader,
        criterion,
        device,
        desc=f"Testing {name}"
    )

    true_scores = np.array(true_labels) + 1
    pred_scores = np.array(pred_labels) + 1

    within_1_acc = np.mean(np.abs(true_scores - pred_scores) <= 1)
    mae = np.mean(np.abs(true_scores - pred_scores))

    print("Loss:", test_loss)
    print(f"Accuracy：{test_acc * 100:.2f}%")
    print(f"Macro-F1：{test_f1 * 100:.2f}%")
    print(f"±1 分內 Accuracy：{within_1_acc * 100:.2f}%")
    print(f"MAE 平均誤差：{mae:.4f}")

    print("\n分類報告：")
    print(classification_report(
        true_labels,
        pred_labels,
        target_names=["1分", "2分", "3分", "4分", "5分"],
        digits=4
    ))

    print("\n混淆矩陣：")
    print(confusion_matrix(true_labels, pred_labels))

    all_test_results[name] = {
        "loss": test_loss,
        "accuracy": test_acc,
        "macro_f1": test_f1,
        "within_1_acc": within_1_acc,
        "mae": mae,
        "true_labels": true_labels,
        "pred_labels": pred_labels
    }



========== 測試資料：women ==========


Testing women: 100%|██████████| 99/99 [00:01<00:00, 57.92it/s]


Loss: 1.3172027233875159
Accuracy：49.35%
Macro-F1：38.71%
±1 分內 Accuracy：86.20%
MAE 平均誤差：0.6946

分類報告：
              precision    recall  f1-score   support

          1分     0.1967    0.4502    0.2738       211
          2分     0.2533    0.4203    0.3161       414
          3分     0.2571    0.3517    0.2970       799
          4分     0.3284    0.3917    0.3573      1422
          5分     0.8558    0.5797    0.6912      3450

    accuracy                         0.4935      6296
   macro avg     0.3783    0.4387    0.3871      6296
weighted avg     0.5990    0.4935    0.5271      6296


混淆矩陣：
[[  95   80   28    4    4]
 [ 118  174   86   26   10]
 [ 122  249  281  102   45]
 [  69  118  400  557  278]
 [  79   66  298 1007 2000]]


========== 測試資料：walmart ==========


Testing walmart: 100%|██████████| 385/385 [00:06<00:00, 58.64it/s]

Loss: 1.1139313820120575
Accuracy：59.16%
Macro-F1：44.45%
±1 分內 Accuracy：87.89%
MAE 平均誤差：0.5985

分類報告：
              precision    recall  f1-score   support

          1分     0.6018    0.6459    0.6231      2971
          2分     0.2070    0.2710    0.2347      1133
          3分     0.1958    0.3391    0.2483      1501
          4分     0.3013    0.4480    0.3603      3893
          5分     0.8726    0.6670    0.7561     15119

    accuracy                         0.5916     24617
   macro avg     0.4357    0.4742    0.4445     24617
weighted avg     0.6776    0.5916    0.6225     24617


混淆矩陣：
[[ 1919   510   286   104   152]
 [  406   307   235   100    85]
 [  286   274   509   296   136]
 [  178   165   706  1744  1100]
 [  400   227   863  3544 10085]]


In [15]:
for name, df in test_dfs.items():
    true_labels = all_test_results[name]["true_labels"]
    pred_labels = all_test_results[name]["pred_labels"]

    result_df = df.copy()
    result_df["true_score"] = np.array(true_labels) + 1
    result_df["pred_score"] = np.array(pred_labels) + 1
    result_df["correct"] = result_df["true_score"] == result_df["pred_score"]
    result_df["error"] = result_df["pred_score"] - result_df["true_score"]

    output_file = f"{name}_prediction_result_macro_f1_model.csv"
    result_df.to_csv(output_file, index=False, encoding="utf-8-sig")

    print(f"已輸出：{output_file}")
    display(result_df[[TEXT_COL, "true_score", "pred_score", "correct", "error"]].head(10))

已輸出：women_prediction_result_macro_f1_model.csv


,Text,true_score,pred_score,correct,error
0,absolutely wonderful silky and sexy and comfor...,4,5,False,1
1,love this dress it s sooo pretty i happened to...,5,5,True,0
2,i had such high hopes for this dress and reall...,3,2,False,-1
3,i love love love this jumpsuit it s fun flirty...,5,5,True,0
4,this shirt is very flattering to all due to th...,5,5,True,0
5,i love tracy reese dresses but this one is not...,2,2,True,0
6,i aded this in my basket at hte last mintue to...,5,4,False,-1
7,i ordered this in carbon for store pick up and...,4,4,True,0
8,i love this dress i usually get an xs but it r...,5,5,True,0
9,i m 5 5 and 125 lbs i ordered the s petite to ...,5,4,False,-1


已輸出：walmart_prediction_result_macro_f1_model.csv


,Text,true_score,pred_score,correct,error
0,one star for looking nice thats it after tryin...,1,1,True,0
1,love this phone so far have had it almost a mo...,4,4,True,0
2,this tv is absolutely fantastic this is my thi...,4,5,False,1
3,refurb good shape good price does what i want ...,5,4,False,-1
4,very nice tablet looks brand new fired right u...,5,5,True,0
5,i purchased it a few months ago because my son...,5,5,True,0
6,great sound quality highly recommend great for...,5,5,True,0
8,cheap headphones with extremely good sound qua...,5,5,True,0
10,the installation was very easy just as the adv...,4,4,True,0
11,item was advertised by walmart as refurbished ...,4,1,False,-3


In [16]:
import torch
import numpy as np

def predict_score(review_text):
    model.eval()

    # 1. 清理文字
    review_text_clean = clean_text(review_text)

    # 2. 轉成模型吃的數字序列
    ids = encode_text(review_text_clean, word2idx, MAX_LEN)

    # 3. 轉成 tensor，放到 GPU / CPU
    input_ids = torch.tensor([ids], dtype=torch.long).to(device)

    # 4. 預測
    with torch.no_grad():
        logits = model(input_ids)
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()

    # 5. label 0~4 轉回 Score 1~5
    pred_label = int(np.argmax(probs))
    pred_score = pred_label + 1

    return pred_score, probs

print("predict_score 函式已建立，可以開始輸入評論測試。")

predict_score 函式已建立，可以開始輸入評論測試。


In [21]:
# =========================
# 1. 類別 × 使用年限修正表
# =========================

category_age_adjustment = {
    "家具寢具": {
        "1年內": 0.3,
        "1-3年": 0.1,
        "3-5年": -0.2,
        "5-8年": -0.6,
        "8以上": -1.0,
    },
    "家電與3C": {
        "1年內": 0.4,
        "1-3年": 0.1,
        "3-5年": -0.4,
        "5-8年": -0.9,
        "8以上": -1.3,
    },
    "服飾配件": {
        "1年內": 0.3,
        "1-3年": 0.0,
        "3-5年": -0.4,
        "5-8年": -0.8,
        "8以上": -1.2,
    },
    "美妝個護": {
        "1年內": 0.2,
        "1-3年": -0.4,
        "3-5年": -1.0,
        "5-8年": -1.5,
        "8以上": -2.0,
    },
    "書籍辦公": {
        "1年內": 0.2,
        "1-3年": 0.1,
        "3-5年": 0.0,
        "5-8年": -0.2,
        "8以上": -0.4,
    },
    "生活廚具": {
        "1年內": 0.3,
        "1-3年": 0.0,
        "3-5年": -0.3,
        "5-8年": -0.7,
        "8以上": -1.1,
    },
    "運動愛好": {
        "1年內": 0.3,
        "1-3年": 0.0,
        "3-5年": -0.4,
        "5-8年": -0.8,
        "8以上": -1.2,
    },
    "紀念品": {
        "1年內": 0.4,
        "1-3年": 0.3,
        "3-5年": 0.2,
        "5-8年": 0.1,
        "8以上": 0.0,
    },
}


# =========================
# 2. 類別 + 使用年限修正
# =========================

def category_age_adjust(category, age_group):
    if category not in category_age_adjustment:
        raise ValueError(
            f"未知類別：{category}。請輸入：{list(category_age_adjustment.keys())}"
        )

    if age_group not in category_age_adjustment[category]:
        raise ValueError(
            f"未知使用年限：{age_group}。請輸入：{list(category_age_adjustment[category].keys())}"
        )

    return category_age_adjustment[category][age_group]


# =========================
# 3. 物品狀況分數修正
# =========================

def condition_adjustment(condition_score):
    """
    物品狀況分數 1~5

    1 = 狀況很差
    2 = 狀況偏差
    3 = 普通
    4 = 狀況良好
    5 = 狀況很好
    """

    condition_score = int(condition_score)

    table = {
        1: -0.70,
        2: -0.35,
        3: 0.00,
        4: 0.35,
        5: 0.70
    }

    if condition_score not in table:
        raise ValueError("物品狀況分數只能輸入 1、2、3、4、5")

    return table[condition_score]


# =========================
# 4. 使用頻率修正
# =========================

def frequency_adjustment(category, frequency_score):
    """
    使用頻率 1~5

    1 = 幾乎沒使用
    2 = 偶爾使用
    3 = 普通使用
    4 = 經常使用
    5 = 非常常使用

    規則：
    家具寢具 / 家電與3C / 生活廚具 / 運動愛好：
        使用越頻繁，磨損越高，分數越低。

    其他類別：
        使用越頻繁，代表越實用或越有價值，分數越高。
    """

    frequency_score = int(frequency_score)

    if frequency_score not in [1, 2, 3, 4, 5]:
        raise ValueError("使用頻率只能輸入 1、2、3、4、5")

    wear_sensitive_categories = [
        "家具寢具",
        "家電與3C",
        "生活廚具",
        "運動愛好"
    ]

    if category in wear_sensitive_categories:
        # 使用越高，扣越多
        table = {
            1: 0.30,
            2: 0.10,
            3: 0.00,
            4: -0.30,
            5: -0.60
        }
    else:
        # 使用越高，加越多
        table = {
            1: -0.20,
            2: 0.00,
            3: 0.20,
            4: 0.40,
            5: 0.60
        }

    return table[frequency_score]


# =========================
# 5. 從模型 base_score 微調最終分數
# =========================

def adjust_score_from_base(
    base_score,
    category,
    age_group,
    condition_score,
    frequency_score
):
    """
    base_score:
        模型根據文字描述預測出的分數

    category:
        家具寢具 / 家電與3C / 服飾配件 / 美妝個護 /
        書籍辦公 / 生活廚具 / 運動愛好 / 紀念品

    age_group:
        1年內 / 1-3年 / 3-5年 / 5-8年 / 8以上

    condition_score:
        物品狀況分數 1~5

    frequency_score:
        使用頻率 1~5

    最後分數：
        不限制在 1~5
        保留小數
    """

    age_adj = category_age_adjust(category, age_group)
    cond_adj = condition_adjustment(condition_score)
    freq_adj = frequency_adjustment(category, frequency_score)

    final_score = base_score + age_adj + cond_adj + freq_adj

    return {
        "base_score": base_score,
        "category": category,
        "age_group": age_group,
        "condition_score": condition_score,
        "frequency_score": frequency_score,
        "category_age_adjustment": age_adj,
        "condition_adjustment": cond_adj,
        "frequency_adjustment": freq_adj,
        "final_score": final_score
    }




In [ ]:
# =========================
# 6. 單筆互動式輸入版本
# =========================

while True:
    my_review = input("請輸入物品狀況描述，輸入 q 離開：")

    if my_review.lower() == "q":
        print("結束測試")
        break

    # 1. 模型先根據文字描述預測 base_score
    score, probs = predict_score(my_review)

    print("\n========== 模型原始預測 ==========")
    print("模型 base_score：", score)

    print("\n模型各分數機率：")
    for i, p in enumerate(probs, start=1):
        print(f"{i} 分：{p:.4f}")

    # 2. 使用者輸入額外資訊
    print("\n可選類別：")
    print("家具寢具 / 家電與3C / 服飾配件 / 美妝個護 / 書籍辦公 / 生活廚具 / 運動愛好 / 紀念品")
    category = input("請輸入物品類別：").strip()

    print("\n可選使用年限：")
    print("1年內 / 1-3年 / 3-5年 / 5-8年 / 8以上")
    age_group = input("請輸入使用年限：").strip()

    condition_score = input("請輸入物品狀況分數 1~5：").strip()
    frequency_score = input("請輸入使用頻率 1~5，1=幾乎沒使用，5=非常常使用：").strip()

    # 3. 規則微調
    try:
        result = adjust_score_from_base(
            base_score=score,
            category=category,
            age_group=age_group,
            condition_score=condition_score,
            frequency_score=frequency_score
        )

        print("\n========== 加減分後結果 ==========")
        print("模型 base_score：", result["base_score"])
        print("類別年限修正：", result["category_age_adjustment"])
        print("物品狀況修正：", result["condition_adjustment"])
        print("使用頻率修正：", result["frequency_adjustment"])

        print("\n計算公式：")
        print(
            f"{result['base_score']} "
            f"+ ({result['category_age_adjustment']}) "
            f"+ ({result['condition_adjustment']}) "
            f"+ ({result['frequency_adjustment']})"
        )

        print("\n最後小數分數：", round(result["final_score"], 2))

    except ValueError as e:
        print("\n輸入錯誤：", e)

    print("-" * 50)

請輸入物品狀況描述，輸入 q 離開： Noise amplification function is normal



========== 模型原始預測 ==========
模型 base_score： 5

模型各分數機率：
1 分：0.0832
2 分：0.0661
3 分：0.0977
4 分：0.1037
5 分：0.6494

可選類別：
家具寢具 / 家電與3C / 服飾配件 / 美妝個護 / 書籍辦公 / 生活廚具 / 運動愛好 / 紀念品


請輸入物品類別： 家電與3C



可選使用年限：
1年內 / 1-3年 / 3-5年 / 5-8年 / 8以上


請輸入使用年限： 3-5年
請輸入物品狀況分數 1~5： 3
請輸入使用頻率 1~5，1=幾乎沒使用，5=非常常使用： 5


...


可選使用年限：
1年內 / 1-3年 / 3-5年 / 5-8年 / 8以上


請輸入使用年限： 1-3年
請輸入物品狀況分數 1~5： 3
請輸入使用頻率 1~5，1=幾乎沒使用，5=非常常使用： 5



========== 加減分後結果 ==========
模型 base_score： 4
類別年限修正： 0.0
物品狀況修正： 0.0
使用頻率修正： 0.6

計算公式：
4 + (0.0) + (0.0) + (0.6)

最後小數分數： 4.6
--------------------------------------------------


請輸入物品狀況描述，輸入 q 離開： Its exterior was tattered and worn, but it felt very nice to the touch.



========== 模型原始預測 ==========
模型 base_score： 4

模型各分數機率：
1 分：0.0189
2 分：0.0209
3 分：0.0961
4 分：0.6800
5 分：0.1842

可選類別：
家具寢具 / 家電與3C / 服飾配件 / 美妝個護 / 書籍辦公 / 生活廚具 / 運動愛好 / 紀念品
